# CoNLL-2003 Named Entity Recognition with PyTorch

This notebook is designed for Google Colab and implements an RNN-based NER model using the CoNLL-2003 dataset.

**Contents:**

1. Setup and imports
2. Load and inspect CoNLL-2003 dataset
3. Preprocessing and vectorization
4. PyTorch dataset and dataloader
5. BiLSTM sequence labeling model
6. Training, validation, and evaluation
7. Improvements and next steps


In [2]:
# Install dependencies when running in Colab
!pip install -q torch torchvision torchaudio datasets seqeval optuna


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 40.3 MB/s eta 0:00:00


In [3]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from seqeval.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score,
)
import optuna
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [4]:
# Load the CoNLL-2003 dataset
dataset = load_dataset("BramVanroy/conll2003")

print(dataset)
print("Train examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["validation"]))
print("Test examples:", len(dataset["test"]))

# Inspect one example
example = dataset["train"][0]
print("Tokens:", example["tokens"])
print("NER tags:", example["ner_tags"])
print("Chunk tags:", example["chunk_tags"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/310k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/281k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})
Train examples: 14041
Validation examples: 3250
Test examples: 3453
Tokens: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
NER tags: [3, 0, 7, 0, 0, 0, 7, 0, 0]
Chunk tags: [11, 21, 11, 12, 21, 22, 11, 12, 0]


In [5]:
# Tag maps and labels
ner_feature = dataset["train"].features["ner_tags"]
ner_feature.feature.int2str

<bound method ClassLabel.int2str of ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'])>

In [6]:
id2label = ner_feature.feature.int2str
label2id = {label: idx for idx, label in enumerate(ner_feature.feature.names)}
pos_feature = dataset["train"].features["pos_tags"]
chunk_feature = dataset["train"].features["chunk_tags"]
id2pos = pos_feature.feature.int2str
id2chunk = chunk_feature.feature.int2str

print("Label set:", ner_feature.feature.names)
print("POS set:", pos_feature.feature.names)
print("Chunk set:", chunk_feature.feature.names)

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_LABEL = "<PAD_LABEL>"

POS_PAD = "<PAD_POS>"
CHUNK_PAD = "<PAD_CHUNK>"


def normalize_token(token):
    return token.lower()


def parse_sentences(split_dataset):
    sentences = []
    for item in split_dataset:
        tokens = [normalize_token(t) for t in item["tokens"]]
        tags = [id2label(tag) for tag in item["ner_tags"]]
        pos_tags = [id2pos(tag) for tag in item["pos_tags"]]
        chunk_tags = [id2chunk(tag) for tag in item["chunk_tags"]]
        sentences.append((tokens, tags, pos_tags, chunk_tags))
    return sentences


train_sentences = parse_sentences(dataset["train"])
val_sentences = parse_sentences(dataset["validation"])
test_sentences = parse_sentences(dataset["test"])

print("Sample parsed sentence:", train_sentences[0])

Label set: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
POS set: ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']
Chunk set: ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']
Sample parsed sentence: (['eu', 'rejects', 'german', 'call', 'to', 'boycott', 'british', 'lamb', '.'], ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O'], ['NNP', 'VBZ', 'JJ', 'NN', 'TO', 'VB', 'JJ', 'NN', '.'], ['B-NP', 'B-VP', 'B-NP', 'I-NP', 'B-VP', 'I-VP', 'B-NP', 'I-NP', 'O'])


In [7]:
# Build vocabulary and label maps


def build_vocab(sentences, min_freq=1):
    counter = Counter(token for tokens, _, _, _ in sentences for token in tokens)
    tokens = [PAD_TOKEN, UNK_TOKEN] + [
        word for word, freq in counter.items() if freq >= min_freq
    ]
    word2idx = {word: idx for idx, word in enumerate(tokens)}
    return word2idx


word2idx = build_vocab(train_sentences, min_freq=1)
pos2idx = {
    POS_PAD: 0,
    **{label: idx + 1 for idx, label in enumerate(pos_feature.feature.names)},
}
chunk2idx = {
    CHUNK_PAD: 0,
    **{label: idx + 1 for idx, label in enumerate(chunk_feature.feature.names)},
}
label2idx = {PAD_LABEL: 0, **{label: idx + 1 for idx, label in enumerate(sorted(ner_feature.feature.names))}}
idx2label = {idx: label for label, idx in label2idx.items()}

print("Vocabulary size:", len(word2idx))
print("POS vocab size:", len(pos2idx))
print("Chunk vocab size:", len(chunk2idx))
print("Label map:", label2idx)


def encode_sentence(
    tokens, tags, pos_tags, chunk_tags, word2idx, label2idx, pos2idx, chunk2idx
):
    input_ids = [word2idx.get(token, word2idx[UNK_TOKEN]) for token in tokens]
    tag_ids = [label2idx[tag] for tag in tags]
    pos_ids = [pos2idx[tag] for tag in pos_tags]
    chunk_ids = [chunk2idx[tag] for tag in chunk_tags]
    return input_ids, tag_ids, pos_ids, chunk_ids


def pad_batch(batch, pad_token_id=0, pad_label_id=None, pad_pos_id=0, pad_chunk_id=0):
    if pad_label_id is None:
        pad_label_id = label2idx[PAD_LABEL]
    max_len = max(len(ids) for ids, tags, pos, chunk in batch)
    input_ids = [
        ids + [pad_token_id] * (max_len - len(ids)) for ids, tags, pos, chunk in batch
    ]
    tag_ids = [
        tags + [pad_label_id] * (max_len - len(tags)) for ids, tags, pos, chunk in batch
    ]
    pos_ids = [
        pos + [pad_pos_id] * (max_len - len(pos)) for ids, tags, pos, chunk in batch
    ]
    chunk_ids = [
        chunk + [pad_chunk_id] * (max_len - len(chunk))
        for ids, tags, pos, chunk in batch
    ]
    attention_mask = [
        [1] * len(ids) + [0] * (max_len - len(ids)) for ids, tags, pos, chunk in batch
    ]
    return (
        torch.tensor(input_ids, dtype=torch.long),
        torch.tensor(tag_ids, dtype=torch.long),
        torch.tensor(pos_ids, dtype=torch.long),
        torch.tensor(chunk_ids, dtype=torch.long),
        torch.tensor(attention_mask, dtype=torch.long),
    )


# Class weights for NER labels to reduce O-dominance


def build_label_weights(sentences, label2idx):
    counter = Counter(label for _, tags, _, _ in sentences for label in tags)
    total = sum(counter.values())
    weights = torch.ones(len(label2idx), dtype=torch.float)
    for label, count in counter.items():
        weights[label2idx[label]] = total / (count + 1e-8)
    weights = weights / weights.mean()
    weights[label2idx[PAD_LABEL]] = 0.0
    return weights


label_weights = build_label_weights(train_sentences, label2idx)
print("Label weights:", label_weights)

Vocabulary size: 21011
POS vocab size: 48
Chunk vocab size: 24
Label map: {'<PAD_LABEL>': 0, 'B-LOC': 1, 'B-MISC': 2, 'B-ORG': 3, 'B-PER': 4, 'I-LOC': 5, 'I-MISC': 6, 'I-ORG': 7, 'I-PER': 8, 'O': 9}
Label weights: tensor([0.0000, 0.4712, 0.9786, 0.5322, 0.5097, 2.9078, 2.9128, 0.9083, 0.7430,
        0.0198])


In [8]:
class NERDataset(Dataset):
    def __init__(self, sentences, word2idx, label2idx, pos2idx, chunk2idx):
        self.sentences = sentences
        self.word2idx = word2idx
        self.label2idx = label2idx
        self.pos2idx = pos2idx
        self.chunk2idx = chunk2idx

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        tokens, tags, pos_tags, chunk_tags = self.sentences[idx]
        return encode_sentence(
            tokens,
            tags,
            pos_tags,
            chunk_tags,
            self.word2idx,
            self.label2idx,
            self.pos2idx,
            self.chunk2idx,
        )


def collate_fn(batch):
    input_ids, tag_ids, pos_ids, chunk_ids = zip(*batch)
    return pad_batch(
        list(zip(input_ids, tag_ids, pos_ids, chunk_ids)),
        pad_token_id=word2idx[PAD_TOKEN],
        pad_label_id=label2idx[PAD_LABEL],
    )


train_dataset = NERDataset(train_sentences, word2idx, label2idx, pos2idx, chunk2idx)
val_dataset = NERDataset(val_sentences, word2idx, label2idx, pos2idx, chunk2idx)
test_dataset = NERDataset(test_sentences, word2idx, label2idx, pos2idx, chunk2idx)

BATCH_SIZE = 32
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

print("Example batch sizes:")
for batch in train_loader:
    print([x.shape for x in batch])
    break

Example batch sizes:
[torch.Size([32, 47]), torch.Size([32, 47]), torch.Size([32, 47]), torch.Size([32, 47]), torch.Size([32, 47])]


In [9]:
# Download and load GloVe embeddings
os.system('wget -q -O glove.6B.zip http://nlp.stanford.edu/data/glove.6B.zip')
os.system('unzip -q -o glove.6B.zip glove.6B.100d.txt')


0

In [10]:


def load_glove_embeddings(glove_path, word2idx, embedding_dim=100):
    matrix = np.random.normal(scale=0.1, size=(len(word2idx), embedding_dim)).astype(np.float32)
    matrix[word2idx[PAD_TOKEN]] = np.zeros(embedding_dim, dtype=np.float32)
    with open(glove_path, encoding='utf-8') as f:
        for line in f:
            values = line.strip().split()
            if len(values) != embedding_dim + 1:
                continue
            word = values[0]
            vector = np.asarray(values[1:], dtype=np.float32)
            if word in word2idx:
                matrix[word2idx[word]] = vector
    return matrix

EMBEDDING_DIM = 100
POS_EMBEDDING_DIM = 25
CHUNK_EMBEDDING_DIM = 25
HIDDEN_DIM = 128
NUM_LABELS = len(label2idx)

glove_path = 'glove.6B.100d.txt'
embedding_matrix = load_glove_embeddings(glove_path, word2idx, EMBEDDING_DIM)


In [11]:

class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, embedding_dim, pos_vocab_size, pos_embedding_dim, chunk_vocab_size, chunk_embedding_dim, hidden_dim, num_labels,num_layers, dropout=0.3):
        super().__init__()
        self.word_embedding = nn.Embedding.from_pretrained(
            torch.from_numpy(embedding_matrix), freeze=False, padding_idx=word2idx[PAD_TOKEN]
        )
        self.pos_embedding = nn.Embedding(pos_vocab_size, pos_embedding_dim, padding_idx=pos2idx[POS_PAD])
        self.chunk_embedding = nn.Embedding(chunk_vocab_size, chunk_embedding_dim, padding_idx=chunk2idx[CHUNK_PAD])
        self.lstm = nn.LSTM(
            embedding_dim + pos_embedding_dim + chunk_embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, input_ids, pos_ids, chunk_ids, attention_mask=None):
        word_emb = self.word_embedding(input_ids)
        pos_emb = self.pos_embedding(pos_ids)
        chunk_emb = self.chunk_embedding(chunk_ids)
        embeddings = torch.cat([word_emb, pos_emb, chunk_emb], dim=-1)
        lengths = attention_mask.sum(1).cpu() if attention_mask is not None else None
        packed = (
            torch.nn.utils.rnn.pack_padded_sequence(
                embeddings, lengths, batch_first=True, enforce_sorted=False
            )
            if lengths is not None
            else embeddings
        )
        outputs, _ = self.lstm(packed) if lengths is not None else self.lstm(embeddings)
        if lengths is not None:
            outputs, _ = torch.nn.utils.rnn.pad_packed_sequence(outputs, batch_first=True)
        outputs = self.dropout(outputs)
        logits = self.classifier(outputs)
        return logits


In [12]:

model = BiLSTMTagger(
    len(word2idx),
    EMBEDDING_DIM,
    len(pos2idx),
    POS_EMBEDDING_DIM,
    len(chunk2idx),
    CHUNK_EMBEDDING_DIM,
    HIDDEN_DIM,
    NUM_LABELS,
    num_layers=3
).to(device)

criterion = nn.CrossEntropyLoss(weight=label_weights.to(device), ignore_index=label2idx[PAD_LABEL])
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

print(model)


BiLSTMTagger(
  (word_embedding): Embedding(21011, 100, padding_idx=0)
  (pos_embedding): Embedding(48, 25, padding_idx=0)
  (chunk_embedding): Embedding(24, 25, padding_idx=0)
  (lstm): LSTM(150, 128, num_layers=3, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=256, out_features=10, bias=True)
)


In [13]:
class EarlyStopping:
    def __init__(self, patience=3, delta=0.0, path='best_model.pth', verbose=False):
        self.patience = patience
        self.delta = delta
        self.path = path
        self.verbose = verbose
        self.best_score = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, score, model):
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.path)
        if self.verbose:
            print(f'Saved best model to {self.path}')

early_stopping = EarlyStopping(patience=3, delta=0.0, path='best_ner_model.pth', verbose=True)


In [14]:

def train_epoch(model, data_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for input_ids, tag_ids, pos_ids, chunk_ids, attention_mask in data_loader:
        input_ids = input_ids.to(device)
        tag_ids = tag_ids.to(device)
        pos_ids = pos_ids.to(device)
        chunk_ids = chunk_ids.to(device)
        attention_mask = attention_mask.to(device)

        optimizer.zero_grad()
        logits = model(input_ids, pos_ids, chunk_ids, attention_mask)
        loss = criterion(logits.view(-1, NUM_LABELS), tag_ids.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(data_loader)


def evaluate(model, data_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for input_ids, tag_ids, pos_ids, chunk_ids, attention_mask in data_loader:
            input_ids = input_ids.to(device)
            tag_ids = tag_ids.to(device)
            pos_ids = pos_ids.to(device)
            chunk_ids = chunk_ids.to(device)
            attention_mask = attention_mask.to(device)
            logits = model(input_ids, pos_ids, chunk_ids, attention_mask)
            predictions = logits.argmax(-1).cpu().numpy()
            labels = tag_ids.cpu().numpy()
            mask = attention_mask.cpu().numpy()
            batch_size, seq_len = predictions.shape
            for i in range(batch_size):
                preds = [
                    idx2label[predictions[i][j]]
                    for j in range(seq_len)
                    if mask[i][j] == 1
                ]
                refs = [
                    idx2label[labels[i][j]] for j in range(seq_len) if mask[i][j] == 1
                ]
                all_preds.append(preds)
                all_labels.append(refs)
    return all_preds, all_labels

def fit(model, train_loader, val_loader, optimizer, criterion, device, early_stopping, epochs=20):
    best_val_f1 = 0.0
    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_preds, val_labels = evaluate(model, val_loader, device)
        val_f1 = f1_score(val_labels, val_preds)

        print(f'Epoch {epoch}: train_loss={train_loss:.4f} val_f1={val_f1:.4f}')

        early_stopping(val_f1, model)
        if early_stopping.early_stop:
            print('Early stopping triggered')
            break

    model.load_state_dict(torch.load('best_ner_model.pth', map_location=device))
    return model, best_val_f1

model, best_val_f1 = fit(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    early_stopping,
    epochs=20,
)
model.load_state_dict(torch.load('best_ner_model.pth'))

print("Best validation F1:", best_val_f1)

Epoch 1: train_loss=0.7337 val_f1=0.6544
Saved best model to best_ner_model.pth
Epoch 2: train_loss=0.2678 val_f1=0.7365
Saved best model to best_ner_model.pth
Epoch 3: train_loss=0.1556 val_f1=0.7484
Saved best model to best_ner_model.pth
Epoch 4: train_loss=0.0954 val_f1=0.7617
Saved best model to best_ner_model.pth
Epoch 5: train_loss=0.0626 val_f1=0.8037
Saved best model to best_ner_model.pth
Epoch 6: train_loss=0.0462 val_f1=0.8358
Saved best model to best_ner_model.pth
Epoch 7: train_loss=0.0364 val_f1=0.7562
EarlyStopping counter: 1 out of 3
Epoch 8: train_loss=0.0358 val_f1=0.7935
EarlyStopping counter: 2 out of 3
Epoch 9: train_loss=0.0301 val_f1=0.8314
EarlyStopping counter: 3 out of 3
Early stopping triggered
Best validation F1: 0.0


In [15]:
# Final evaluation on the test dataset
model.load_state_dict(torch.load("best_ner_model.pth", map_location=device))
test_preds, test_labels = evaluate(model, test_loader, device)
print("Test precision:", precision_score(test_labels, test_preds))
print("Test recall:", recall_score(test_labels, test_preds))
print("Test F1:", f1_score(test_labels, test_preds))

print(classification_report(test_labels, test_preds))

Test precision: 0.7527003063034016
Test recall: 0.8266643059490085
Test F1: 0.7879503839338453
              precision    recall  f1-score   support

         LOC       0.88      0.88      0.88      1668
        MISC       0.53      0.64      0.58       702
         ORG       0.66      0.82      0.73      1661
         PER       0.85      0.86      0.86      1617

   micro avg       0.75      0.83      0.79      5648
   macro avg       0.73      0.80      0.76      5648
weighted avg       0.76      0.83      0.79      5648



In [16]:
# Hyperparameter tuning with Optuna


def objective(trial):
    hidden_dim = trial.suggest_int("hidden_dim", 128, 256, step=64)
    dropout = trial.suggest_float("dropout", 0.2, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-3, log=True)
    number_layers = trial.suggest_int("num_layers",1,5)

    model = BiLSTMTagger(
        len(word2idx),
        EMBEDDING_DIM,
        len(pos2idx),
        POS_EMBEDDING_DIM,
        len(chunk2idx),
        CHUNK_EMBEDDING_DIM,
        hidden_dim,
        NUM_LABELS,
        dropout=dropout,
        num_layers=number_layers
    ).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    criterion = nn.CrossEntropyLoss(
        weight=label_weights.to(device), ignore_index=label2idx[PAD_LABEL]
    )

    for epoch in range(1, 5):
        train_epoch(model, train_loader, optimizer, criterion, device)
    val_preds, val_labels = evaluate(model, val_loader, device)
    return f1_score(val_labels, val_preds)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Best trial:", study.best_trial.params)
print("Best validation F1:", study.best_value)

[I 2026-04-21 20:57:50,087] A new study created in memory with name: no-name-fefa3ee1-ab6b-4815-8275-badefbed6b42
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2927965687126934 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
[I 2026-04-21 20:58:11,927] Trial 0 finished with value: 0.6278836509528586 and parameters: {'hidden_dim': 256, 'dropout': 0.2927965687126934, 'lr': 0.00024863715079913594, 'num_layers': 1}. Best is trial 0 with value: 0.6278836509528586.
[I 2026-04-21 20:58:39,966] Trial 1 finished with value: 0.5811032222829055 and parameters: {'hidden_dim': 128, 'dropout': 0.20245388371648085, 'lr': 0.00024756997631481036, 'num_layers': 2}. Best is trial 0 with value: 0.6278836509528586.
[I 2026-04-21 20:59:33,865] Trial 2 finished with value: 0.5084053002834729 and parameters: {'hidden_dim': 256

Best trial: {'hidden_dim': 128, 'dropout': 0.31415857559966803, 'lr': 0.0009164077144304572, 'num_layers': 4}
Best validation F1: 0.7677146879399344


In [22]:


model = BiLSTMTagger(
    len(word2idx),
    EMBEDDING_DIM,
    len(pos2idx),
    POS_EMBEDDING_DIM,
    len(chunk2idx),
    CHUNK_EMBEDDING_DIM,
    hidden_dim=study.best_trial.params['hidden_dim'],
    num_labels=NUM_LABELS,
    num_layers=study.best_trial.params['num_layers'],
    dropout=study.best_trial.params['dropout'],
).to(device);

criterion = nn.CrossEntropyLoss(weight=label_weights.to(device), ignore_index=label2idx[PAD_LABEL]);
optimizer = optim.AdamW(model.parameters(), lr=study.best_trial.params['lr'], weight_decay=1e-2);

print(model)




BiLSTMTagger(
  (word_embedding): Embedding(21011, 100, padding_idx=0)
  (pos_embedding): Embedding(48, 25, padding_idx=0)
  (chunk_embedding): Embedding(24, 25, padding_idx=0)
  (lstm): LSTM(150, 128, num_layers=4, batch_first=True, dropout=0.31415857559966803, bidirectional=True)
  (dropout): Dropout(p=0.31415857559966803, inplace=False)
  (classifier): Linear(in_features=256, out_features=10, bias=True)
)


In [23]:

early_stopping = EarlyStopping(patience=3, delta=0.0, path='best_ner_model.pth', verbose=True)

best_model, best_val_f1 = fit(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    early_stopping,
    epochs=20,
)

Epoch 1: train_loss=0.8509 val_f1=0.6420
Saved best model to best_ner_model.pth
Epoch 2: train_loss=0.2932 val_f1=0.7336
Saved best model to best_ner_model.pth
Epoch 3: train_loss=0.1747 val_f1=0.7401
Saved best model to best_ner_model.pth
Epoch 4: train_loss=0.1231 val_f1=0.8167
Saved best model to best_ner_model.pth
Epoch 5: train_loss=0.0830 val_f1=0.8053
EarlyStopping counter: 1 out of 3
Epoch 6: train_loss=0.0605 val_f1=0.8325
Saved best model to best_ner_model.pth
Epoch 7: train_loss=0.0453 val_f1=0.7811
EarlyStopping counter: 1 out of 3
Epoch 8: train_loss=0.0532 val_f1=0.8422
Saved best model to best_ner_model.pth
Epoch 9: train_loss=0.0409 val_f1=0.8284
EarlyStopping counter: 1 out of 3
Epoch 10: train_loss=0.0383 val_f1=0.8202
EarlyStopping counter: 2 out of 3
Epoch 11: train_loss=0.0322 val_f1=0.8516
Saved best model to best_ner_model.pth
Epoch 12: train_loss=0.0270 val_f1=0.7905
EarlyStopping counter: 1 out of 3
Epoch 13: train_loss=0.0175 val_f1=0.8667
Saved best model to 

## Improvements and next steps

- Use GloVe pretrained embeddings that are now integrated into the model.
- Use POS tags and chunk tags as additional embedding features for richer input.
- Apply weighted class cross-entropy to reduce O-tag dominance.
- Perform hyperparameter tuning with Optuna for hidden size, dropout, and learning rate.
- Consider adding a CRF layer for structured BIO decoding and using more epochs.
- Save the final model and export inference helper functions for deployment.
